In [ ]:
%load_ext autoreload
%autoreload 2

from spatial_tcr.utils import setup_plotting, switch_cwd_to_root

switch_cwd_to_root()

from spatial_tcr.pl import utils as plot_utils

figure_dir = "figures/revision/figure-4"
setup_plotting(figure_dir, display_dpi=300, save_dpi=300)

import os

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns

from spatial_tcr.pl import plot_spatial

In [ ]:
path = "data/xenium/processed/05.2-kidney_tcr_infilrate.h5ad"
adata = sc.read_h5ad(path)
adata.obs["cell_id"] = adata.obs.index.str.split("output").str[0]
adata = adata[adata.obs["condition"] == "ANCA"].copy()
adata

In [ ]:
adata.obs["sample"].unique().tolist()

In [ ]:
adata.obs["infiltrate+glom+gdT"] = adata.obs["tcell_infiltrate"].astype(str)
mapping = {"infiltrate": "T cell infiltrate", "no infiltrate": "other"}
adata.obs["infiltrate+glom+gdT"] = adata.obs["infiltrate+glom+gdT"].map(mapping)

# add glom annotations
adata.obs.loc[adata.obs["in_glom"], "infiltrate+glom+gdT"] = "inside glom."

# add gdT annotations
adata.obs.loc[adata.obs["cell_type_l2"] == "gdT", "infiltrate+glom+gdT"] = "gdT"

categories = ["other", "T cell infiltrate", "inside glom.", "gdT"]

# reorder categories
adata.obs["infiltrate+glom+gdT"] = adata.obs["infiltrate+glom+gdT"].astype("category")
adata.obs["infiltrate+glom+gdT"] = adata.obs[
    "infiltrate+glom+gdT"
].cat.reorder_categories(categories)
adata.obs["infiltrate+glom+gdT"].value_counts()

In [ ]:
sample = adata.obs["sample"].unique()[3]

ad_sub = adata[adata.obs["sample"] == sample].copy()

In [ ]:
from spatial_tcr.colors import colors_sub

palette = sns.color_palette("deep", 4)
palette = dict(zip(categories, palette))
palette["gdT"] = "red"

palette = {
    "gdT": colors_sub["gdT"],
    "inside glom.": colors_sub["glom. cell"],
    "other": "#c5c5c5",
    "T cell infiltrate": "#76a9e2",
}
plot_utils.plot_palette_dict(palette)

In [ ]:
fig = sc.pl.spatial(
    ad_sub, color="infiltrate+glom+gdT", spot_size=10, show=False, palette=palette
)
ax = fig[0]
ad_sub_gdt = ad_sub[ad_sub.obs["cell_type_l2"] == "gdT"]
ax.scatter(
    ad_sub_gdt.obsm["spatial"][:, 0],
    ad_sub_gdt.obsm["spatial"][:, 1],
    s=5,
    c=palette["gdT"],
)
ax.set_title("gdT cell distribution around T cell infiltrates and gloms")
sns.despine(ax=ax)
plt.tight_layout()
# plt.savefig(os.path.join(figure_dir, "gdT_infiltrate_glom_distribution.pdf"), dpi=300)

## New plot with zoom in

In [ ]:
base_dir = "/bonn-epyc/projects/spatialTCR/20240719__094819__human_kidney_7_TCR"

samples = adata.obs["sample"].unique()
sample_to_data_dir = dict(
    zip(samples, [os.path.join(base_dir, f"output-{sample}") for sample in samples])
)
cc_to_sample = adata.obs[["cc", "sample"]].apply(np.array).drop_duplicates().values
cc_to_sample = dict(zip(cc_to_sample[:, 0], cc_to_sample[:, 1]))

cc_to_data_dir = {cc: sample_to_data_dir[sample] for cc, sample in cc_to_sample.items()}

In [ ]:
# sample_key = "cc"
# sample = "17"
sample_key = "sample"
data_dir = sample_to_data_dir[sample]

ad_sub = plot_utils.add_spatial_data(
    adata,
    data_dir,
    sample_key=sample_key,
    sample=sample,
    cell_id_key="cell_id",
    level=0,
)
ad_sub.obs_names = ad_sub.obs["cell_id"].tolist()

In [ ]:
plot_col = "infiltrate+glom+gdT"

In [ ]:
fig, ax = plot_spatial(
    ad_sub,
    # plot_nuc=True,
    plot_nuc=False,
    plot_cell=True,
    plot_img=False,
    # genes=["PECAM1"],
    dpi=300,
    cell_color=plot_col,
    # img_kwargs={"vmax": [3000, 5000, 3000]},
    img_kwargs={"vmax": 5000},
    show=False,
    # gene_kwargs={"s": 1},
    cell_kwargs={
        "palette": palette,
        "edgecolor": "none",
        "rasterized": True,
    },
    figsize=(5, 10),
    labelsize=3,
    tick_size=1000,
    img_channel=[
        # "CD45",
        # "RNA",
        "DAPI",
    ],
    grid=True,
    frameon=False,
    show_scale_bar=True,
    scale_bar_kwargs={"color": "black"},
    show_legend=True,
)
# highlight gdTs
mask = ad_sub.obs["cell_type_l2"] == "gdT"
x, y = ad_sub[mask].obsm["spatial_scaled"].T
ax.scatter(x, y, s=2.5, c=palette["gdT"])

plt.savefig(
    os.path.join(figure_dir, "gdT_infiltrate_glom_distribution.pdf"),
    dpi=1000,
    bbox_inches="tight",
)

### Plot zoom in

In [ ]:
palette_alpha = {
    ct: mcolors.to_rgba(palette[ct], alpha=0.0 if ct == "NA" else 0.8)
    for ct in palette.keys()
}
palette_alpha["gdT"] = palette["gdT"]

In [ ]:
xlim = [2500, 2800]
ylim = [800, 1050]

ad_zoom_1 = plot_utils.add_spatial_data(
    ad_sub,
    data_dir,
    xlim=xlim,
    ylim=ylim,
    cell_id_key="cell_id",
    level=0,
)
ad_zoom_1.obs_names = ad_zoom_1.obs["cell_id"].tolist()

fig, ax, legend = plot_spatial(
    ad_zoom_1,
    plot_cell=True,
    dpi=300,
    cell_color=plot_col,
    # genes=genes,
    img_kwargs={"vmax": 5000},
    show=False,
    # gene_kwargs={"s": 30, "palette": gene_palette},
    cell_kwargs={
        "palette": palette_alpha,
        "edgecolor": "lightgray",
        "linewidth": 0.2,
    },
    figsize=(5, 10),
    labelsize=5,
    tick_size=50,
    img_channel=[
        # "CD45",
        # "RNA",
        "DAPI",
    ],
    grid=False,
    frameon=False,
    show_scale_bar=True,
    scale_bar_kwargs={"color": "black"},
    show_legend=True,
    return_legend=True,
)
# highlight gdTs
# mask = ad_zoom_1.obs["cell_type_l2"] == "gdT"
# x, y = ad_zoom_1[mask].obsm["spatial_scaled"].T
# ax.scatter(x, y, s=2.5, c=palette_alpha["gdT"])
plt.savefig(
    os.path.join(figure_dir, "gdT_infiltrate_glom_distribution_zoom.pdf"),
    dpi=300,
    bbox_inches="tight",
)